In [1]:
import os
import re
import pandas as pd
os.chdir("../..")
EVAL_ROOT = "outputs/evaluation"
TRAIN_COMPLEXITY = {
    "protocol_I":  "Binary",
    "protocol_II": "Ternary",
    "protocol_III": "Binary + Ternary",
    "protocol_IV": "Binary + Ternary + ACI set 1",
}

In [2]:
metric_pattern = re.compile(r"(.+?):\s+([0-9eE\.\+\-]+)(?:\s*\+/-\s*([0-9eE\.\+\-]+))?")

SINGLE_VALUE_KEYS = {
    "GD_Loss",
    "Mean Uncertainty",
    "Median Uncertainty",
    "Std Uncertainty",
}

def parse_metrics(file_path):
    out = {}
    with open(file_path, "r") as f:
        for line in f:
            m = metric_pattern.match(line.strip())
            if not m:
                continue

            key, mean, std = m.groups()
            key = key.strip()
            k = key.lower().replace(" ", "_")
            if key in SINGLE_VALUE_KEYS:
                out[k] = float(mean)
                continue
            if std is None:
                out[f"{k}_mean"] = float(mean)
            else:
                out[f"{k}_mean"] = float(mean)
                out[f"{k}_std"]  = float(std)

    return out


In [3]:
rows = []

for protocol in os.listdir(EVAL_ROOT):
    proto_path = os.path.join(EVAL_ROOT, protocol)
    if not os.path.isdir(proto_path):
        continue

    train_complexity = TRAIN_COMPLEXITY.get(protocol, protocol)

    for test_set in os.listdir(proto_path):
        test_path = os.path.join(proto_path, test_set)
        if not os.path.isdir(test_path):
            continue

        test_label = test_set.replace("_", " ").replace("set", "set ").strip()

        for ensemble_folder in os.listdir(test_path):
            ens_path = os.path.join(test_path, ensemble_folder)
            if not os.path.isdir(ens_path):
                continue

            m = re.match(r"Ensemble_(.+?)_constraint", ensemble_folder)
            if m:
                constraint = m.group(1)
            else:
                constraint = "unknown"

            metrics_file = os.path.join(ens_path, "metrics.txt")
            if not os.path.isfile(metrics_file):
                continue

            metrics = parse_metrics(metrics_file)

            row = {
                "Train complexity": train_complexity,
                "Constraint": constraint.capitalize(),
                "Test": test_label,
                }
            row.update(metrics)
            rows.append(row)

In [4]:
df = pd.DataFrame(rows)
protocol_order = [
    "Binary",
    "Ternary",
    "Binary + Ternary",
    "Binary + Ternary + ACI set 1",]

constraint_order = [
    "None",
    "Soft",
    "Hard",]

df["Train complexity"] = pd.Categorical(
    df["Train complexity"],
    categories=protocol_order,
    ordered=True)

df["Constraint"] = pd.Categorical(
    df["Constraint"],
    categories=constraint_order,
    ordered=True)

# ----------------------------
# Test ordering
# ----------------------------
def test_sort_key(label):
    s = label.lower()

    if "infinite dilution" in s:
        m = re.search(r"set\s*(\d+)", s)
        num = int(m.group(1)) if m else 999
        return (2, num)

    if "binary" in s:
        return (1, 0)

    return (0, 0)

df["Test_rank"] = df["Test"].apply(test_sort_key)

df = df.sort_values(
    ["Train complexity", "Constraint", "Test_rank"]
).reset_index(drop=True)

df = df.drop(columns=["Test_rank"])

output_path = "outputs/evaluation/metrics_summary.csv"
df.to_csv(output_path, index=False)
